# W4.3 — Thiết kế GAT và Check-in 1

**Chủ sở hữu:** B. Phạm vi tuần 2–4, độc lập branch A.

**Input:** Summary W2.1/W2.3/W3.1/W4.1/W4.2; không tự chạy lại notebook thí nghiệm.

Hợp đồng tuần 6: `forward(x, train_adjacency, return_attention=False)` trả embedding;
khi yêu cầu attention trả `(embedding, (processed_edges, alpha))`. Thêm self-loop đúng một lần trước attention.
Không dùng full graph khi tuning LP. Decoder dot-product nhận evaluation pairs riêng.

| Bước | Shape | Quy ước |
|---|---|---|
| Feature dropout | [N,F] | Chỉ train mode |
| Linear W không bias | [N,K,F_prime] | K head |
| LeakyReLU attention score | [E,K] | source j, target i; slope 0.2 |
| Softmax theo target | [E,K] | Trừ max từng nhóm/head |
| Attention dropout | [E,K] | Sau softmax, chỉ train mode |
| Weighted sum | [N,K,F_prime] | Cộng theo node nhận |
| Concat hoặc mean | [N,K*F_prime] hoặc [N,F_prime] | Hidden concat; output mean |
| Bias rồi activation | như output | ELU hidden; output identity |

Attention trả về là alpha trước dropout để kiểm tra tổng bằng 1; message dùng alpha sau dropout.
Tuần này chỉ thử grouped softmax, chưa ghép thành GAT hoàn chỉnh.

In [ ]:
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p/'configs').is_dir() and (p/'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở Jupyter từ repo hoặc thư mục con có configs/ và notebooks/.')

In [ ]:
import numpy as np
import torch
from utils.experiment import (
    read_json,
    write_json,
    seed_everything,
    array_hash,
    mapping_hash,
    environment,
    new_run,
    publish_result,
    expect_error,
)


In [ ]:
config = read_json(PROJECT_ROOT/'configs/training_demo.json')
print(config)

{'dataset': 'toy', 'model': 'linear_regression', 'task': 'regression', 'seed': 42, 'device': 'cpu', 'learning_rate': 0.05, 'weight_decay': 0.0, 'max_epochs': 150, 'patience': 12, 'monitor': 'validation_mse', 'mode': 'min', 'min_delta': 1e-06}


In [ ]:
from torch_geometric.utils import softmax as pyg_softmax

from utils.attention import receiver_softmax

scores = torch.tensor([[10000.,-10000.],[10001.,-9998.],[7.,9.]],dtype=torch.float64,requires_grad=True)
targets = torch.tensor([1,1,2],dtype=torch.long)
alpha = receiver_softmax(scores,targets,4)
torch.testing.assert_close(alpha,pyg_softmax(scores,targets,num_nodes=4))
torch.testing.assert_close(alpha[:2].sum(0),torch.ones(2,dtype=alpha.dtype))
torch.testing.assert_close(alpha[2],torch.ones(2,dtype=alpha.dtype))
alpha.square().sum().backward()
assert torch.isfinite(scores.grad).all() and scores.grad.abs().sum()>0
assert receiver_softmax(torch.empty((0,2)),torch.empty(0,dtype=torch.long),4).shape == (0,2)
# Kiểm tra shape concat/mean không cần tạo GAT model.
projected = torch.zeros(4,3,2)
assert projected.flatten(1).shape == (4,6) and projected.mean(1).shape == (4,2)
checks = {'grouped_softmax_multhead_large_single_gradient':{'status':'passed','alpha':alpha.detach().tolist()},
          'concat_mean_shape':{'status':'passed','concat':[4,6],'mean':[4,2]}}
checks['reject_bad_target'] = expect_error(lambda:receiver_softmax(scores.detach(),torch.tensor([1,1,4]),4))

D:\gnn-node-link-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\gnn-node-link-prediction\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
eda = read_json(PROJECT_ROOT/'results/week02/eda_summary.json')
audit = read_json(PROJECT_ROOT/'results/week02/leakage_audit.json')
baseline = read_json(PROJECT_ROOT/'results/week03/cosine_validation.json')
graph = read_json(PROJECT_ROOT/'results/week04/graph_checks.json')
training = read_json(PROJECT_ROOT/'results/week04/training_checks.json')
manifest = read_json(PROJECT_ROOT/'artifacts/splits/cora_lp_v1/manifest.json')
assert audit['split_hash'] == baseline['split_hash'] == manifest['split_hash']
assert eda['input_fingerprints'] == audit['input_fingerprints'] == baseline['input_fingerprints']
assert audit['status'] == graph['status'] == training['status'] == 'passed'
assert baseline['test_value'] is None
for prefix,part in [('graph',graph),('training',training)]:
    checks.update({prefix+'/'+k:v for k,v in part['checks'].items()})
checks['summary_provenance'] = {'status':'passed','cora_split_hash':manifest['split_hash'],
                               'source_runs':{r['task']:r['run_id'] for r in [eda,audit,baseline,graph,training]}}
run = new_run(PROJECT_ROOT,'w4_checkin',config)
result = publish_result(PROJECT_ROOT,run,'results/week04/checks.json',{'status':'passed','checks':checks,
    'pending':['B tự đọc/chạy/giải thích lại','GAT hoàn chỉnh ở tuần 6'],
    'cora_validation':baseline['validation'],'cora_split_hash':manifest['split_hash']},config,4,'W4.3','checkin','toy')
print('Check-in:',result['status']); print('checks:',len(checks)); print('run:',run.name)

Check-in: passed
checks: 16
run: 20260928T070206_w4_checkin_265c4ce5


## Kết luận và tự kiểm tra

Chỉ các assertion chạy thành công mới tạo kết quả bàn giao. Output bên trên chứa đường dẫn/run ID để truy vết. B chưa được đánh dấu đã tự review.

1. Vì sao softmax theo target, không theo toàn bộ E?
2. Dropout attention có giữ tổng bằng 1 không?
3. Concat và mean thay đổi shape ra sao?
4. Run nào chứng minh từng kết quả Check-in?